# Subtitles_gen

Notebook Colab pour lancer Dual Subtitles depuis Google Drive.


## 1. Monter Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Cloner ou mettre a jour le repo dans Drive

Le clone est fait uniquement si le repo n'existe pas deja dans Google Drive.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/git-haddadz/Dual_Subtitles.git"
REPO_DIR = Path("/content/drive/MyDrive/Dual_Subtitles")

if (REPO_DIR / ".git").exists():
    print(f"Repo deja present: {REPO_DIR}")
    !git -C "{REPO_DIR}" pull
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    !git clone "{REPO_URL}" "{REPO_DIR}"

%cd {REPO_DIR}


## 3. Installer ffmpeg et les dependances Python


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

%pip install -q --upgrade pip

# Colab preinstalle souvent une stack torch/transformers trop recente via uv.
# On la retire explicitement avant de reposer les versions du notebook original.
%pip uninstall -y torch torchaudio torchvision transformers tokenizers pyannote.audio pyannote.core pyannote.database pyannote.metrics pyannote.pipeline huggingface_hub
%pip install --no-cache-dir --force-reinstall -r requirements.txt
%pip install -q -e . --no-deps

from importlib import metadata

from pyannote.audio import Pipeline

expected_versions = {
    "torch": "2.2.2",
    "torchaudio": "2.2.2",
    "torchvision": "0.17.2",
    "transformers": "4.41.2",
    "tokenizers": "0.19.1",
    "pyannote.audio": "3.1.1",
    "huggingface-hub": "0.36.2",
    "scipy": "1.11.4",
}

for package_name, expected_version in expected_versions.items():
    installed_version = metadata.version(package_name)
    print(f"{package_name}: {installed_version}")
    assert installed_version.startswith(expected_version), (
        package_name,
        installed_version,
        expected_version,
    )

print("pyannote Pipeline import ok:", Pipeline)


## 4. Ajouter le token Hugging Face

Le token reste en memoire Colab via `HUGGINGFACE_TOKEN`; il n'est pas ecrit dans Drive.


In [ ]:
import getpass
import os

hf_token = getpass.getpass("Hugging Face token: ")
os.environ["HUGGINGFACE_TOKEN"] = hf_token

try:
    from huggingface_hub import login

    login(token=hf_token)
except Exception as exc:
    print(f"Hugging Face login skipped: {exc}")


## 5. Configurer le dossier des videos



In [ ]:
SOURCE_FOLDER = Path("")
OUTPUT_FOLDER = SOURCE_FOLDER
TEMP_FOLDER = Path("")

SOURCE_FOLDER.mkdir(parents=True, exist_ok=True)
TEMP_FOLDER.mkdir(parents=True, exist_ok=True)

mp4_files = sorted(SOURCE_FOLDER.glob("*.mp4"))
print(f"Found {len(mp4_files)} mp4 file(s) in {SOURCE_FOLDER}")
for video in mp4_files:
    print("-", video.name)


## 6. Generer les sous-titres SRT et ASS


In [ ]:
import logging
import sys

repo_src = REPO_DIR / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from dual_subtitles.core.config import ProcessingConfig
from dual_subtitles.core.pipeline import process_directory

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

config = ProcessingConfig(
    input_dir=SOURCE_FOLDER,
    output_dir=OUTPUT_FOLDER,
    temp_dir=TEMP_FOLDER,
    transcription_language="ar",
    translation_source_language="ar",
    translation_target_language="en",
    whisper_model="openai/whisper-large-v3",
    diarization_model="pyannote/speaker-diarization-3.1",
    generate_srt=True,
    generate_ass=True,
    skip_existing=True,
    use_diarization=True,
)

generated_files = process_directory(config)
print("Generated files:")
for path in generated_files:
    print("-", path)
